# Macaron food photo model

This notebook trains the small model that lets the Macaron app recognise a dish from a photo, on the phone, with no internet.

**What it does**

1. Downloads two public food photo datasets.
2. Fine-tunes a small pretrained network (MobileNetV3) to tell the dishes apart.
3. Saves two files for the app: `food.onnx` (the model) and `labels.json` (the dish names).

**How to run it on Kaggle**

1. Go to kaggle.com, sign in, and choose **Create** then **New Notebook**.
2. **File** then **Import Notebook**, and upload this file.
3. In the right-hand panel, open **Session options**. Set **Accelerator** to **GPU T4 x2** and **Internet** to **On**.
4. Press **Run All** at the top. It takes roughly 30 to 60 minutes.
5. When it finishes, open the **Output** section on the right. Download `food.onnx` and `labels.json`.
6. Put both files in the `model` folder of the app's repository.

The model can only recognise the dishes listed in `MAP` below. Each entry links a dataset class to the exact dish name used in the app.

In [ ]:
!pip -q install timm datasets onnx onnxruntime

In [ ]:
import json, random, numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms as T
from datasets import load_dataset
from PIL import Image

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", DEVICE)

# dataset class -> exact dish name in the app's food list
MAP_MY = {  # lowyisan/malaysian_food_images
    "fish_and_chips": "Fish and chips", "fried_noodles": "Mee goreng mamak", "fried_rice": "Nasi goreng kampung",
    "hamburger": "Cheeseburger", "kaya_toast": "Kaya toast set", "laksa": "Curry laksa",
    "mixed_rice": "Economy rice: 1 meat, 2 veg", "nasi_lemak": "Nasi lemak biasa", "popiah": "Popiah, fresh",
    "roti_canai": "Roti canai kosong", "satay": "Satay ayam",
}
MAP_F101 = {  # ethz/food101, only the classes that are common in Malaysia and exist in the app
    "sushi": "Sushi plates", "sashimi": "Salmon sashimi and rice", "ramen": "Ramen, shoyu", "bibimbap": "Bibimbap",
    "pad_thai": "Pad thai", "pizza": "Pizza", "spaghetti_bolognese": "Spaghetti bolognese",
    "spaghetti_carbonara": "Spaghetti carbonara", "caesar_salad": "Chicken caesar salad", "french_fries": "French fries",
    "ice_cream": "Ice cream", "chocolate_cake": "Chocolate cake", "cheesecake": "Cheesecake", "donuts": "Doughnut",
    "hummus": "Hummus and falafel plate", "falafel": "Hummus and falafel plate", "edamame": "Edamame",
    "miso_soup": "Miso soup", "chicken_wings": "Fried chicken", "dumplings": "Har gow",
    "omelette": "Omelette with rice and cucumber", "grilled_salmon": "Pan-seared salmon, rice and greens",
    "gyoza": "Gyoza", "takoyaki": "Takoyaki", "pho": "Pho", "spring_rolls": "Spring rolls, fried",
    "chicken_curry": "Chicken curry with rice", "peking_duck": "Roast duck rice", "waffles": "Waffle with syrup",
    "pancakes": "Pancakes with syrup", "french_toast": "French toast", "club_sandwich": "Club sandwich",
    "steak": "Steak with vegetables", "hot_dog": "Hot dog", "tiramisu": "Tiramisu", "lasagna": "Lasagna",
    "macaroni_and_cheese": "Macaroni and cheese", "garlic_bread": "Garlic bread", "samosa": "Samosa",
    "cup_cakes": "Cupcake", "frozen_yogurt": "Frozen yogurt", "tacos": "Tacos", "macarons": "Macarons",
    "fried_calamari": "Fried calamari", "greek_salad": "Garden salad", "eggs_benedict": "Eggs Benedict",
    "apple_pie": "Apple pie",
}
MAP_IN = {  # rajistics/indian_food_images (small, so it only tops up these classes)
    "chapati": "Chapati with dhal", "idli": "Idli with sambar", "masala_dosa": "Masala thosai",
    "chai": "Teh tarik", "samosa": "Samosa",
}
PER_CLASS = 500      # cap on training photos per class, to keep classes balanced
EPOCHS_HEAD, EPOCHS_ALL = 2, 8
BATCH = 64

## Load the photos

Both datasets come from the Hugging Face Hub. Food-101 is large (about 5 GB), so this step takes a few minutes.

In [ ]:
my = load_dataset("lowyisan/malaysian_food_images")
f101 = load_dataset("ethz/food101")

my_names = my["train"].features["label"].names
f_names = f101["train"].features["label"].names

try:
    india = load_dataset("rajistics/indian_food_images")
    in_names = india["train"].features["label"].names
except Exception as e:                                  # optional extra data: carry on without it
    india = None; MAP_IN = {}; print("Skipping the Indian food dataset:", e)

CLASSES = sorted(set(MAP_MY) | set(MAP_F101) | set(MAP_IN))   # model output order
IDX = {c: i for i, c in enumerate(CLASSES)}
print(len(CLASSES), "classes")

def pick(ds, names, mapping, cap, seed=SEED):
    """Row indices for the wanted classes, at most `cap` per class."""
    labels = ds["label"]                                # reads the label column only, no image decoding
    by = {}
    for i, l in enumerate(labels):
        n = names[l]
        if n in mapping: by.setdefault(n, []).append(i)
    rng = random.Random(seed); out = []
    for n, idx in by.items():
        rng.shuffle(idx); out += [(i, IDX[n]) for i in idx[:cap]]
    return out

class Photos(Dataset):
    def __init__(self, ds, rows, tf): self.ds, self.rows, self.tf = ds, rows, tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, k):
        i, y = self.rows[k]
        return self.tf(self.ds[i]["image"].convert("RGB")), y

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def square(img):
    """Centre square crop then resize to 224. The app prepares photos exactly this way."""
    w, h = img.size; m = min(w, h)
    return img.crop(((w - m) // 2, (h - m) // 2, (w + m) // 2, (h + m) // 2)).resize((224, 224), Image.BILINEAR)

train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.55, 1.0)), T.RandomHorizontalFlip(),
                      T.ColorJitter(0.3, 0.3, 0.3, 0.05), T.ToTensor(), T.Normalize(MEAN, STD)])
test_tf = T.Compose([T.Lambda(square), T.ToTensor(), T.Normalize(MEAN, STD)])

train_ds = ConcatDataset([Photos(my["train"], pick(my["train"], my_names, MAP_MY, PER_CLASS), train_tf),
                          Photos(f101["train"], pick(f101["train"], f_names, MAP_F101, PER_CLASS), train_tf)]
                         + ([Photos(india["train"], pick(india["train"], in_names, MAP_IN, PER_CLASS), train_tf)] if india else []))
val_ds = ConcatDataset([Photos(my["test"], pick(my["test"], my_names, MAP_MY, 150), test_tf),
                        Photos(f101["validation"], pick(f101["validation"], f_names, MAP_F101, 150), test_tf)])
print(len(train_ds), "training photos,", len(val_ds), "test photos")

train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=4, drop_last=True, pin_memory=True)
val_dl = DataLoader(val_ds, BATCH, shuffle=False, num_workers=4, pin_memory=True)

## Train

First only the new final layer is trained, then the whole network at a lower learning rate.

In [ ]:
model = timm.create_model("mobilenetv3_large_100", pretrained=True, num_classes=len(CLASSES)).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")

def evaluate():
    model.eval(); right = np.zeros(len(CLASSES)); seen = np.zeros(len(CLASSES))
    with torch.no_grad():
        for x, y in val_dl:
            p = model(x.to(DEVICE)).argmax(1).cpu()
            for a, b in zip(p.tolist(), y.tolist()): seen[b] += 1; right[b] += a == b
    return right.sum() / seen.sum(), right / np.maximum(seen, 1)

def run(epochs, lr, head_only):
    for n, p in model.named_parameters(): p.requires_grad = (not head_only) or n.startswith("classifier")
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.02)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl))
    for ep in range(epochs):
        model.train(); total = 0
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, enabled=DEVICE == "cuda"):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); total += loss.item()
        acc, _ = evaluate()
        print(f"epoch {ep + 1}/{epochs}  loss {total / len(train_dl):.3f}  test accuracy {acc:.1%}")

run(EPOCHS_HEAD, 2e-3, head_only=True)
run(EPOCHS_ALL, 4e-4, head_only=False)

acc, per = evaluate()
print(f"\nFinal test accuracy: {acc:.1%}\n")
for c, a in sorted(zip(CLASSES, per), key=lambda t: t[1]): print(f"{a:6.1%}  {c}")

## Save the two files for the app

In [ ]:
import onnx, onnxruntime as ort

model.eval().cpu()
dummy = torch.zeros(1, 3, 224, 224)
kw = dict(input_names=["input"], output_names=["logits"], opset_version=17)
try: torch.onnx.export(model, dummy, "food.onnx", dynamo=False, **kw)
except TypeError: torch.onnx.export(model, dummy, "food.onnx", **kw)
onnx.checker.check_model(onnx.load("food.onnx"))

MAP = {**MAP_IN, **MAP_MY, **MAP_F101}
json.dump([{"label": c, "food": MAP[c]} for c in CLASSES], open("labels.json", "w"), indent=1)

# check the saved model gives the same answers as the trained one
sess = ort.InferenceSession("food.onnx")
x, y = next(iter(val_dl))
a = sess.run(None, {"input": x[:1].numpy()})[0]
b = model(x[:1]).detach().numpy()
print("largest difference between saved and trained model:", float(np.abs(a - b).max()))

import os
print(f"food.onnx: {os.path.getsize('food.onnx') / 1e6:.1f} MB.  labels.json: {len(CLASSES)} classes.")
print("Download both files from the Output panel and put them in the app's model folder.")

## Adding your own dishes later

To teach the model a dish that is not in the list:

1. Collect at least 80 to 100 of your own photos of it, taken the way you normally would.
2. Add them as another dataset class, add a line to `MAP` with the exact dish name from the app's food list, and run the notebook again.

Only use photos you took yourself or that are published under a licence that allows reuse.